In [49]:
from pathlib import Path
from scripts.data_loading import read_all
import pandas as pd
import numpy as np

In [50]:
REPO_ROOT = Path.cwd().parent
TRAIN = REPO_ROOT / "data" / "train"

train_input = read_all(TRAIN, "input_*.csv")
train_output = read_all(TRAIN, "output_*.csv")

train/input_*.csv: found 18 files
train/output_*.csv: found 18 files


In [51]:
print(train_input["player_height"].unique())

<ArrowStringArray>
[ '6-1',  '6-0', '5-11',  '6-4',  '6-3', '5-10',  '5-9',  '5-8',  '6-2',
  '6-5',  '6-6',  '5-7',  '6-7',  '6-8',  '5-6',  '6-9']
Length: 16, dtype: str


In [52]:
h = train_input["player_height"].str.split("-", expand=True).astype(int)
train_input["height_in"] = h[0] * 12 + h[1]

In [53]:
print(train_input["height_in"].unique())

[73 72 71 76 75 70 69 68 74 77 78 67 79 80 66 81]


In [54]:
print(train_input["player_side"].unique())

<ArrowStringArray>
['Defense', 'Offense']
Length: 2, dtype: str


In [55]:
print(train_input["player_role"].unique())

<ArrowStringArray>
['Defensive Coverage', 'Other Route Runner', 'Passer', 'Targeted Receiver']
Length: 4, dtype: str


In [56]:
train_input["is_offense"] = (train_input["player_side"] == "Offense").astype(int)
train_input["is_target"]  = (train_input["player_role"] == "Targeted Receiver").astype(int)

# play_direction is used for geometry, see below
# player_position (19 values): one-hot, or group into a few classes (QB, WR, CB, S, LB, ...)
pos = pd.get_dummies(train_input["player_position"], prefix="pos", dtype=int)
train_input = pd.concat([train_input, pos], axis=1)
roles = pd.get_dummies(train_input["player_role"], prefix="role", dtype=int)
train_input = pd.concat([train_input, roles], axis=1)

In [57]:
print(train_input['x'])

0          52.33
1          52.33
2          52.33
3          52.35
4          52.37
           ...  
4880574    41.61
4880575    40.82
4880576    40.03
4880577    39.24
4880578    38.45
Name: x, Length: 4880579, dtype: float64


In [58]:
print(train_input['play_direction'])

0          right
1          right
2          right
3          right
4          right
           ...  
4880574     left
4880575     left
4880576     left
4880577     left
4880578     left
Name: play_direction, Length: 4880579, dtype: str


In [59]:
def standardize_direction(df):
    df = df.copy()
    left = df["play_direction"] == "left"

    df.loc[left, "x"] = 120 - df.loc[left, "x"]
    df.loc[left, "y"] = 53.3 - df.loc[left, "y"]
    df.loc[left, "ball_land_x"] = 120 - df.loc[left, "ball_land_x"]
    df.loc[left, "ball_land_y"] = 53.3 - df.loc[left, "ball_land_y"]
    df.loc[left, ["o", "dir"]] = (df.loc[left, ["o", "dir"]] + 180) % 360
    return df

In [60]:
print(train_input[["x","y","ball_land_x","ball_land_y"]].max(),'\n', train_input[["x","y","ball_land_x","ball_land_y"]].min())

x              119.860000
y               52.880000
ball_land_x    125.849998
ball_land_y     57.330002
dtype: float64 
 x              0.41
y              0.62
ball_land_x   -5.26
ball_land_y   -3.91
dtype: float64


In [61]:
print(train_input['x'])

0          52.33
1          52.33
2          52.33
3          52.35
4          52.37
           ...  
4880574    41.61
4880575    40.82
4880576    40.03
4880577    39.24
4880578    38.45
Name: x, Length: 4880579, dtype: float64


In [62]:
train_input.info()

<class 'pandas.DataFrame'>
RangeIndex: 4880579 entries, 0 to 4880578
Data columns (total 50 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   game_id                   int64  
 1   play_id                   int64  
 2   player_to_predict         bool   
 3   nfl_id                    int64  
 4   frame_id                  int64  
 5   play_direction            str    
 6   absolute_yardline_number  int64  
 7   player_name               str    
 8   player_height             str    
 9   player_weight             int64  
 10  player_birth_date         str    
 11  player_position           str    
 12  player_side               str    
 13  player_role               str    
 14  x                         float64
 15  y                         float64
 16  s                         float64
 17  a                         float64
 18  dir                       float64
 19  o                         float64
 20  num_frames_output         int64  
 

In [64]:
# last k frames as separate features (lags)
k = 5
g = train_input.groupby(["game_id", "play_id", "nfl_id"])
for i in range(1, k):
    for c in ["x", "y", "s", "a"]:
        train_input[f"{c}_lag{i}"] = g[c].shift(i)
last_k = g.tail(1)   # one row per player, with the lag columns

In [67]:
train_input.info()

<class 'pandas.DataFrame'>
RangeIndex: 4880579 entries, 0 to 4880578
Data columns (total 66 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   game_id                   int64  
 1   play_id                   int64  
 2   player_to_predict         bool   
 3   nfl_id                    int64  
 4   frame_id                  int64  
 5   play_direction            str    
 6   absolute_yardline_number  int64  
 7   player_name               str    
 8   player_height             str    
 9   player_weight             int64  
 10  player_birth_date         str    
 11  player_position           str    
 12  player_side               str    
 13  player_role               str    
 14  x                         float64
 15  y                         float64
 16  s                         float64
 17  a                         float64
 18  dir                       float64
 19  o                         float64
 20  num_frames_output         int64  
 

In [68]:
def split_train_val(inp, out, val_frac=0.2, by="game_id", seed=42):
    """
    Split input/output tables into train and validation sets.
    by="game_id" keeps whole games together (safest); use by="play" to split by play.
    """
    inp = inp.copy()
    out = out.copy()

    if by == "play":
        inp["split_key"] = inp["game_id"].astype(str) + "_" + inp["play_id"].astype(str)
        out["split_key"] = out["game_id"].astype(str) + "_" + out["play_id"].astype(str)
    else:
        inp["split_key"] = inp[by]
        out["split_key"] = out[by]

    keys = inp["split_key"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(keys)

    n_val = int(len(keys) * val_frac)
    val_keys = set(keys[:n_val])

    is_val_in = inp["split_key"].isin(val_keys)
    is_val_out = out["split_key"].isin(val_keys)

    # noinspection shadowing-names
    train_in, val_in = inp[~is_val_in].drop(columns="split_key"), inp[is_val_in].drop(columns="split_key")
    # noinspection shadowing-names
    train_out, val_out = out[~is_val_out].drop(columns="split_key"), out[is_val_out].drop(columns="split_key")

    print(f"train: {train_in[['game_id','play_id']].drop_duplicates().shape[0]} plays | "
          f"val: {val_in[['game_id','play_id']].drop_duplicates().shape[0]} plays")
    return train_in, val_in, train_out, val_out

In [69]:
train_in, val_in, train_out, val_out = split_train_val(train_input, train_output, val_frac=0.2)

train: 11343 plays | val: 2765 plays


In [72]:
val_out.info()

<class 'pandas.DataFrame'>
Index: 110662 entries, 5153 to 559079
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   game_id      110662 non-null  int64  
 1   play_id      110662 non-null  int64  
 2   nfl_id       110662 non-null  int64  
 3   frame_id     110662 non-null  int64  
 4   x            110662 non-null  float64
 5   y            110662 non-null  float64
 6   source_file  110662 non-null  str    
dtypes: float64(2), int64(4), str(1)
memory usage: 8.4 MB
